<a href="https://colab.research.google.com/github/RuhiGohel/Final2/blob/main/F1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Indian E-commerce Returns & Fulfilment Analysis
## Data Cleaning and Preparation

This notebook loads the raw order-item dataset, standardizes columns and data types, checks data quality, and exports a cleaned CSV for analysis.

### 1. Import Libraries

In [ ]:
import pandas as pd
from pathlib import Path

### 2. Load Dataset

In [ ]:
file = Path("raw_india_ecommerce_orders_returns.csv")

if file.exists():
    data_path = file
else:
    %pip install -q kagglehub
    import kagglehub

    folder = Path(kagglehub.dataset_download(
        "abirambharathisa/india-e-commerce-orders-and-returns-2024-2026"
    ))
    files = list(folder.rglob("*.csv"))

    if not files:
        raise FileNotFoundError("No CSV file found.")

    data_path = files[0]

df = pd.read_csv(data_path, engine="python")

print("Source file:", data_path)
print("Dataset Shape:", df.shape)
display(df.head())

### 3. Clean Column Names and Text

In [ ]:
df.columns = (
    df.columns.str.strip()
    .str.lower()
    .str.replace(r"[^a-z0-9]+", "_", regex=True)
    .str.strip("_")
)

for col in df.select_dtypes(include="object").columns:
    df[col] = df[col].str.strip()

### 4. Select Relevant Columns

In [ ]:
keep_cols = [
    "order_id", "order_item_id", "order_datetime", "customer_id",
    "state", "region", "is_plus_member", "payment_method",
    "category", "sub_category", "brand", "product_id", "seller_id",
    "seller_rating", "quantity", "mrp", "discount_pct", "selling_price",
    "item_total", "courier_partner", "promised_delivery_days",
    "shipped_datetime", "delivered_datetime", "actual_delivery_days",
    "is_late_delivery", "order_status", "cancel_reason", "item_status",
    "is_returned", "return_reason", "return_status", "refund_mode",
    "refund_amount", "refund_days"
]

missing_cols = [col for col in keep_cols if col not in df.columns]

if missing_cols:
    raise KeyError(f"Missing columns: {missing_cols}")

df = df[keep_cols].copy()

### 5. Fix Data Types and Remove Duplicates

In [ ]:
date_cols = [
    "order_datetime", "shipped_datetime", "delivered_datetime"
]

for col in date_cols:
    df[col] = pd.to_datetime(df[col], errors="coerce")

numeric_cols = [
    "seller_rating", "quantity", "mrp", "discount_pct",
    "selling_price", "item_total", "promised_delivery_days",
    "actual_delivery_days", "is_late_delivery",
    "refund_amount", "refund_days"
]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")

for col in ["is_plus_member", "is_returned"]:
    values = df[col].astype("string").str.strip().str.lower()
    df[col] = values.map({"true": 1, "false": 0, "1": 1, "0": 0})

df = df.drop_duplicates().reset_index(drop=True)

print("Cleaned Dataset Shape:", df.shape)

### 6. Check Missing Values and Data Quality

In [ ]:
missing = pd.DataFrame({
    "Missing Count": df.isna().sum(),
    "Missing Percentage": (df.isna().mean() * 100).round(2)
})

display(missing[missing["Missing Count"] > 0])

checks = pd.Series({
    "Missing Order IDs": df["order_id"].isna().sum(),
    "Missing Order-Item IDs": df["order_item_id"].isna().sum(),
    "Duplicate Order-Item Keys": df.duplicated(
        ["order_id", "order_item_id"]
    ).sum(),
    "Non-positive Quantities": (df["quantity"] <= 0).sum(),
    "Invalid Discounts": (
        (df["discount_pct"] < 0) | (df["discount_pct"] > 100)
    ).sum()
})

display(checks.to_frame("Result"))

### 7. Export Cleaned Dataset

In [ ]:
output_file = Path("cleaned_ecommerce_order_items.csv")
df.to_csv(output_file, index=False)

print("Cleaned dataset saved:", output_file.resolve())
print("Final Shape:", df.shape)